# PPI++ example

One PPI++ run: same split as ppi_mean.py, compared with classical (lam=0) and PPI (lam=1).

Run from `examples/simulation/`, so that `dataset` and `model` import directly.

In [1]:
from sklearn.ensemble import RandomForestRegressor

from openppi import split, PPIPlusPlus, Mean, PPIData
from dataset.judge_sim import make_judge_data

In [2]:
# 1. Split iid rows; fit only on the independent training sample.
frame = make_judge_data(n=2000, seed=0)
train, labeled, unlabeled = split(
    frame.drop(columns='label').to_numpy(), frame['label'].to_numpy(),
    train_size=400, labeled_size=400, random_state=42)
model = RandomForestRegressor(n_estimators=60, min_samples_leaf=8,
                              max_depth=8, random_state=0, n_jobs=1)
model.fit(*train)
data = PPIData(labels=labeled[1], labeled_predictions=model.predict(labeled[0]),
               unlabeled_predictions=model.predict(unlabeled))

In [3]:
# 2. Select the target.
target = Mean()

In [4]:
# 3. Tune lambda by the plug-in estimate; fixed lambdas give the two baselines.
for label, method in (('classical', PPIPlusPlus(lam=0)), ('PPI', PPIPlusPlus(lam=1)), ('PPI++', PPIPlusPlus())):
    result = method(data, target=target, alpha=.05)
    low, high = result.confidence_interval
    # 4. Output.
    print(f'{label:9s} lambda={result.diagnostics["prediction_weight"]:.3f} '
          f'estimate={result.estimate:.4f} CI=({low:.4f}, {high:.4f}) width={high - low:.4f}')

classical lambda=0.000 estimate=0.5000 CI=(0.4509, 0.5491) width=0.0981
PPI       lambda=1.000 estimate=0.4939 CI=(0.4457, 0.5422) width=0.0965
PPI++     lambda=0.536 estimate=0.4967 CI=(0.4501, 0.5434) width=0.0933
